<a href="https://colab.research.google.com/github/rdiazrincon/conformal_prediction_pd/blob/master/pd_reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PD LEDD Change Prediction — Reconstruction (no `Results_UPDATED.csv`)

Mirrors the original pipeline exactly, excluding only the note-parser that required
`Results_UPDATED.csv`. As a result, `home_meds` is set to `0` for all patients —
this is the only intentional feature difference. Everything else (LED calculation,
time-window aggregation, temporal features, drug one-hots, model hyperparameters)
is identical to the original notebook.

**Expected files in `data/`:**
`TRY_NEW.csv` · `CROSSOVER_2.csv` · `DEMOGRAPHICS.csv` · `DIAGNOSIS_DATE_3.csv` · `DBS_pts.csv`

`data.csv` and `series.txt` are **not** needed here — we reconstruct them from scratch.


## 1 · Imports and constants

In [1]:
import numpy as np
import pandas as pd
import json, re, warnings
from datetime import datetime
from scipy import stats
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)
warnings.filterwarnings('ignore')

RANDOM_STATE  = 21
DATA_CUTOFF   = datetime(2021, 12, 31)
TIME_WINDOW   = '1Y'   # matches the aggregation used to produce the original data.csv
# Stage 2 cutoff is derived from the Youden index on this model's ROC curve (see Stage 1 cell)


## 2 · Load raw files

In [2]:
# ── 2. Load raw files ─────────────────────────────────────────────────────────
# Use CROSSOVER_2.csv as primary drug source — has full visit history with
# visit_start_datetime (outpatient + inpatient), giving ~8 years/patient.
# TRY_NEW.csv is inpatient-only (2-3 dates/patient → only 2-3 yearly buckets).
drug_exposure = pd.read_csv(
    'data/CROSSOVER_2.csv',
    dtype={'dose_unit_source_value': str},
    engine='python',
    on_bad_lines='warn',
)
drug_exposure['drug_exposure_start_datetime'] = pd.to_datetime(drug_exposure['drug_exposure_start_datetime'])
drug_exposure['visit_start_datetime']         = pd.to_datetime(drug_exposure['visit_start_datetime'])
drug_exposure['dose_source_value']            = np.nan
drug_exposure['dose_unit_source_value']       = np.nan
drug_exposure['route_source_value']           = np.nan
drug_exposure['visit_detail_id']              = np.nan

demographics = pd.read_csv('data/DEMOGRAPHICS.csv')
demographics['birth_datetime'] = pd.to_datetime(demographics['birth_datetime'])
demographics['age'] = ((pd.Timestamp('now') - demographics['birth_datetime']).dt.days / 365.25).astype(int)
demographics.drop(columns=['birth_datetime'], inplace=True)
demographics = demographics[['person_id', 'age', 'gender_source_value',
                              'race_source_value', 'ethnicity_source_value']]

diagnosis_date = pd.read_csv('data/DIAGNOSIS_DATE_3.csv')
diagnosis_date['diagnosis_date'] = pd.to_datetime(diagnosis_date['diagnosis_date'])

visit_occurrence = pd.read_csv('data/CROSSOVER_2.csv', engine='python', on_bad_lines='warn')
visit_occurrence['visit_start_datetime'] = pd.to_datetime(visit_occurrence['visit_start_datetime'])

dbs_df = pd.read_csv('data/DBS_pts.csv')
dbs_df['procedure_date']       = pd.to_datetime(dbs_df['procedure_date'])
dbs_df['condition_start_date'] = pd.to_datetime(dbs_df['condition_start_date'])

print(f"drug_exposure    : {drug_exposure['person_id'].nunique()} patients, {len(drug_exposure):,} rows")
print(f"demographics     : {len(demographics):,} rows")
print(f"visit_occurrence : {visit_occurrence['person_id'].nunique()} patients")
print(f"dbs_df           : {dbs_df['person_id'].nunique()} DBS patients")

drug_exposure    : 631 patients, 554,590 rows
demographics     : 2,724 rows
visit_occurrence : 631 patients
dbs_df           : 142 DBS patients


## 3 · Parse drug names and dosages from `TRY_NEW.csv`

In [3]:
# ── 3. Parse drug names and dosages from drug_exposure ────────────────────────
def safe_parse_dsv(val):
    try:
        parsed = json.loads(val)
        name = parsed.get('med_display_name', '')
        return name if isinstance(name, str) else ''
    except Exception:
        return ''

dsv_drug_exposure           = drug_exposure['drug_source_value'].apply(safe_parse_dsv)
drug_info_drug_source_value = {i: v for i, v in enumerate(dsv_drug_exposure)}
drugs_used_drug_exposure    = [v.lower() for v in drug_info_drug_source_value.values()]

drug_names_pattern = r"([\w\s-]+)\s(?:\(([\w\s-]+)\)\s*)?"
dosage_pattern     = r"\d+(?:\.\d+)?(?:-\d+(?:\.\d+)?)*(?:\s*(?:mg/ml|mg|ml))(?:/hr)?"

generic_names_de, brand_names_de, dosages_de = [], [], []
for string in drugs_used_drug_exposure:
    m = re.findall(drug_names_pattern, string)
    if m:
        g, b = m[0]
        generic_names_de.append(g)
        brand_names_de.append(b if b else np.nan)
        dm = re.findall(dosage_pattern, string)
        dosages_de.append(dm[0] if dm else np.nan)
    else:
        generic_names_de.extend([np.nan])
        brand_names_de.extend([np.nan])
        dosages_de.extend([np.nan])

pd_data_drug_exposure = pd.DataFrame({
    'generic_name': generic_names_de,
    'brand_name':   brand_names_de,
    'dosage':       dosages_de,
})

# Known dosage string fixes
for bad, good in [('1mg', '1 mg'), ('200mg', '200 mg')]:
    idx = pd_data_drug_exposure[pd_data_drug_exposure['dosage'] == bad].index
    pd_data_drug_exposure.loc[idx, 'dosage'] = good

print(f"pd_data_drug_exposure: {len(pd_data_drug_exposure):,} rows, "
      f"{pd_data_drug_exposure['generic_name'].nunique()} unique drug names")

pd_data_drug_exposure: 554,590 rows, 1480 unique drug names


## 4 · Build `led_df` and calculate LED

In [4]:
led_dose = []
for item in pd_data_drug_exposure['dosage']:
    if isinstance(item, float) and np.isnan(item):
        led_dose.append('0')
    elif '-' in str(item):
        led_dose.append(str(item).split('-')[1].split()[0])
    else:
        led_dose.append(str(item).split()[0])

led_df = pd.concat(
    [drug_exposure.iloc[:, 0:3], pd_data_drug_exposure, drug_exposure.iloc[:, 3:]], axis=1
)
led_df.insert(loc=6, column='led_dose', value=led_dose)
led_df.insert(loc=3, column='drug_info', value=drug_info_drug_source_value)
led_df['led_dose']                    = pd.to_numeric(led_df['led_dose'], errors='coerce')
led_df['drug_exposure_start_datetime'] = pd.to_datetime(led_df['drug_exposure_start_datetime'])
led_df['generic_name'].replace({np.nan: 'None'}, inplace=True)
led_df['brand_name'].replace({np.nan: 'None'},   inplace=True)

conversion_factors = {
    'amantadine': 1.0,        'amantadine er': 1.25,
    'apomorphine': 10.0,      'benztropine': 1.0,
    'benztropine mesylate': 1.0, 'bromocriptine': 10.0,
    'cabergoline': 66.7,
    'carbidopa-levodopa': 1.0,
    'inv carbidopa-levodopa intestinal gel': 1.0,
    'inv carbidopa-levodopa intestinal gel pump': 1.0,
    'carbidopa': 0.1,          'carbidopa-levodopa er': 0.5,
    'carbidopa-levodopa-entacapone': 1.33, 'entacapone': 1.33,
    'pramipexole': 100.0,      'pramipexole er': 100.0,
    'trihexyphenidyl': 1.0,    'rasagiline': 100.0,
    ' rasagiline mesylate': 100.0,
    'ropinirole': 0.5,         'rotigotine': 30.0,
    'selegiline': 10.0,        'tolcapone': 1.5,
}

def calculate_led(row):
    dsv = row['dose_source_value']
    ld  = row['led_dose']
    cf  = conversion_factors.get(row['generic_name'], 0)
    if ld == dsv:
        dsv = 1.0
    if dsv == 0.0:
        return None
    return dsv * ld * cf if pd.notna(dsv) else ld * cf

led_df['led'] = led_df.apply(calculate_led, axis=1)

# Set LED=NaN for non-PD drugs so they produce empty resample buckets
# that get forward-filled — this preserves temporal coverage from all visits
pd_drug_names = set(conversion_factors.keys())
led_df.loc[~led_df['generic_name'].isin(pd_drug_names), 'led'] = np.nan

print(f"led_df: {led_df['person_id'].nunique()} patients, {len(led_df):,} rows")
print(f"PD drug rows: {led_df['led'].notna().sum():,}  |  non-PD (NaN): {led_df['led'].isna().sum():,}")

led_df: 631 patients, 554,590 rows
PD drug rows: 47,449  |  non-PD (NaN): 507,141


## 5 · Merge `visit_occurrence` into `led_df`

In [5]:
# ── 5. led_df already has visit_start_datetime from CROSSOVER_2.csv ───────────
led_df.sort_values(by='visit_start_datetime', ascending=True, inplace=True)

print(f"led_df: {led_df['person_id'].nunique()} patients, {len(led_df):,} rows")
print(f"visit_start_datetime: {led_df['visit_start_datetime'].min().date()} → {led_df['visit_start_datetime'].max().date()}")
unique_years = led_df.groupby('person_id')['visit_start_datetime'].apply(lambda x: x.dt.year.nunique())
print(f"Unique years per patient: min={unique_years.min()}  median={unique_years.median():.0f}  max={unique_years.max()}")

led_df: 631 patients, 554,590 rows
visit_start_datetime: 2011-05-16 → 2021-04-29
Unique years per patient: min=1  median=4  max=11


## 6 · DBS setup

In [6]:
cpt_codes = [
    '95970','95961','95962','61867','61885','95972','95978','95983','95979','61868',
    '95984','95974','61886','L8681','95971','61888','61880','C1787','95973',
    '00H03MZ','0NH00NZ','00W03MZ','0JWT0MZ','0JPT0MZ','00P00MZ','00W00MZ',
    '00P03MZ','00H00MZ','0JH60BZ','0JH60DZ','00H04MZ',
]
icd_codes = ['Z96.82', 'T85.110A']

procedures = dbs_df['procedure_source_value'].copy().fillna('None')
conditions = dbs_df['condition_source_value'].copy().fillna('None')
for val in procedures.unique():
    for code in cpt_codes:
        if code in str(val): procedures.replace(val, code, inplace=True)
for val in conditions.unique():
    for code in icd_codes:
        if code in str(val): conditions.replace(val, code, inplace=True)
dbs_df['procedure_source_value'] = procedures
dbs_df['condition_source_value'] = conditions

dbs_pts    = dbs_df['person_id'].unique().tolist()
all_pts    = led_df['person_id'].unique().tolist()
has_dbs_df = pd.DataFrame({
    'person_id': all_pts,
    'has_dbs':   [1 if pt in dbs_pts else 0 for pt in all_pts],
})
print(f"DBS patients: {has_dbs_df['has_dbs'].sum()} / {len(has_dbs_df)}")


DBS patients: 142 / 631


## 7 · Time-window aggregation → `mean_led_per_visit`

In [7]:
mean_led_per_visit = (
    led_df.set_index('visit_start_datetime')
          .groupby('person_id')['led']
          .resample(TIME_WINDOW)
          .mean()
          .reset_index()
)
mean_led_per_visit.rename(columns={'drug_exposure_start_datetime': 'visit_start_datetime'}, inplace=True)
mean_led_per_visit = mean_led_per_visit[
    mean_led_per_visit['visit_start_datetime'] <= DATA_CUTOFF
]
mean_led_per_visit.rename(columns={'led': 'mean_led_per_visit'}, inplace=True)

mean_led_per_visit['mean_led_per_visit'] = (
    mean_led_per_visit.groupby('person_id')['mean_led_per_visit']
                      .fillna(method='ffill')
)
mean_led_per_visit['mean_led_per_visit'].fillna(
    mean_led_per_visit['mean_led_per_visit'].mean(), inplace=True
)

print(f"mean_led_per_visit: {mean_led_per_visit['person_id'].nunique()} patients, "
      f"{len(mean_led_per_visit):,} rows")
print(f"Rows per patient (mean): {len(mean_led_per_visit)/mean_led_per_visit['person_id'].nunique():.1f}")

mean_led_per_visit: 631 patients, 3,173 rows
Rows per patient (mean): 5.0


## 8 · Build target variables

In [8]:
mean_led_per_visit['change_in_ledd'] = (
    mean_led_per_visit.groupby('person_id')['mean_led_per_visit'].diff()
)
mean_led_per_visit['percent_change'] = (
    mean_led_per_visit.groupby('person_id')['mean_led_per_visit'].pct_change() * 100
)
mean_led_per_visit['is_first_visit'] = mean_led_per_visit['change_in_ledd'].isna()
mean_led_per_visit.loc[mean_led_per_visit['is_first_visit'], 'change_in_ledd']  = 0
mean_led_per_visit.loc[mean_led_per_visit['is_first_visit'], 'percent_change']  = 0

def normalize_percent_change(series):
    zero_mask   = series == 0
    non_zero    = series[~zero_mask]
    transformed = np.sign(non_zero) * np.log1p(np.abs(non_zero))
    normalized  = stats.mstats.winsorize(transformed, limits=[0.05, 0.05])
    normalized  = (normalized - normalized.min()) / (normalized.max() - normalized.min()) * 2 - 1
    result = pd.Series(index=series.index, dtype=float)
    result[zero_mask]  = 0
    result[~zero_mask] = normalized
    return result

mean_led_per_visit['normalized_percent_change'] = normalize_percent_change(
    mean_led_per_visit['percent_change']
).round(2)
mean_led_per_visit['is_change']  = (mean_led_per_visit['normalized_percent_change'] != 0).astype(int)
mean_led_per_visit['prediction'] = mean_led_per_visit['is_change']

# NOTE: normalized_percent_change is kept in mean_led_per_visit intentionally.
# It is NOT dropped in cleanup — we extract it from xgboost_df in section 13
# so it stays index-aligned with the final feature matrix.

print(f"Class balance  → change: {mean_led_per_visit['prediction'].mean():.1%} "
      f"| no-change: {1 - mean_led_per_visit['prediction'].mean():.1%}")


Class balance  → change: 18.2% | no-change: 81.8%


## 9 · Feature merges

In [9]:
# DBS flag
mean_led_per_visit = mean_led_per_visit.merge(has_dbs_df, on='person_id', how='left')
mean_led_per_visit.sort_values(['person_id', 'visit_start_datetime'],
                                ascending=[False, True], inplace=True)

# Demographics
mean_led_per_visit = mean_led_per_visit.merge(demographics, on='person_id', how='inner')

# Length of stay + days since last visit
los = (
    visit_occurrence.groupby('person_id')['visit_start_datetime']
                    .agg(['min', 'max']).reset_index()
)
los['length_of_stay']       = ((los['max'] - los['min']) / np.timedelta64(1, 'D')).astype(int)
los['days_since_last_visit'] = ((DATA_CUTOFF - los['max']) / np.timedelta64(1, 'D')).astype(int)
los.drop(columns=['min', 'max'], inplace=True)
mean_led_per_visit = mean_led_per_visit.merge(los, on='person_id', how='left')
mean_led_per_visit['length_of_stay'].fillna(los['length_of_stay'].mean(), inplace=True)
mean_led_per_visit['days_since_last_visit'].fillna(los['days_since_last_visit'].mean(), inplace=True)

# Days to PD diagnosis — merge by person_id (avoids positional alignment bug)
first_visit = visit_occurrence.groupby('person_id')['visit_start_datetime'].agg('min').reset_index()
first_diag  = diagnosis_date.groupby('person_id')['diagnosis_date'].agg('min').reset_index()
days_to_dx  = first_visit.merge(first_diag, on='person_id', how='left')
days_to_dx['days_to_diagnosis'] = (
    (days_to_dx['diagnosis_date'] - days_to_dx['visit_start_datetime'])
    / np.timedelta64(1, 'D')
)
days_to_dx['days_to_diagnosis'].fillna(days_to_dx['days_to_diagnosis'].mean(), inplace=True)
days_to_dx['days_to_diagnosis'] = days_to_dx['days_to_diagnosis'].astype(int)
days_to_dx = days_to_dx[['person_id', 'days_to_diagnosis']]
# Use left join to preserve all time-window rows in mean_led_per_visit
mean_led_per_visit = mean_led_per_visit.merge(days_to_dx, on='person_id', how='left')
mean_led_per_visit['days_to_diagnosis'].fillna(days_to_dx['days_to_diagnosis'].mean(), inplace=True)

# Days to / since DBS surgery
dbs_diag = dbs_df.groupby('person_id')[['procedure_date', 'condition_start_date']].agg('min').reset_index()
dbs_diag['dbs_surgery'] = np.where(
    dbs_diag['procedure_date'].isna(), dbs_diag['condition_start_date'],
    np.where(
        dbs_diag['condition_start_date'].isna(), dbs_diag['procedure_date'],
        np.minimum(dbs_diag['procedure_date'], dbs_diag['condition_start_date']),
    ),
)
dbs_diag.drop(columns=['condition_start_date', 'procedure_date'], inplace=True)

days_to_dbs = dbs_diag.merge(first_diag, on='person_id', how='inner')
days_to_dbs.rename(columns={'diagnosis_date': 'pd_diagnosis'}, inplace=True)
days_to_dbs['days_to_dbs']    = ((days_to_dbs['dbs_surgery'] - days_to_dbs['pd_diagnosis']) / np.timedelta64(1, 'D')).astype(int)
days_to_dbs['days_since_dbs'] = ((DATA_CUTOFF - days_to_dbs['dbs_surgery']) / np.timedelta64(1, 'D')).astype(int)
days_to_dbs.drop(columns=['dbs_surgery', 'pd_diagnosis'], inplace=True)

mean_led_per_visit = mean_led_per_visit.merge(days_to_dbs, on='person_id', how='left')
mean_led_per_visit['days_to_dbs'].fillna(days_to_dbs['days_to_dbs'].mean(),       inplace=True)
mean_led_per_visit['days_since_dbs'].fillna(days_to_dbs['days_since_dbs'].mean(), inplace=True)
mean_led_per_visit.drop(columns=['days_to_dbs', 'days_since_dbs'], inplace=True)

# Diagnosed / POA flags (minimal model impact but present in original)
mean_led_per_visit['diagnosed_current_visit'] = mean_led_per_visit.apply(
    lambda row: int(row['person_id'] in
        diagnosis_date[diagnosis_date['diagnosis_date'] == row['visit_start_datetime']]['person_id'].values),
    axis=1,
)
mean_led_per_visit['poa_current_visit'] = mean_led_per_visit.apply(
    lambda row: int(row['person_id'] in
        diagnosis_date[
            (diagnosis_date['condition_poa'] == 1.0) &
            (diagnosis_date['diagnosis_date'] == row['visit_start_datetime'])
        ]['person_id'].values),
    axis=1,
)

# ── Row count checkpoints at each merge ──────────────────────────────────────
n0 = len(mean_led_per_visit)
print(f"Before merges        : {n0:,} rows")

# (merges already happened above — these inspect the current state)
print(f"After all merges     : {mean_led_per_visit['person_id'].nunique()} patients, "
      f"{len(mean_led_per_visit):,} rows")
print(f"NaN counts per column (only columns with NaN):")
nan_cols = mean_led_per_visit.isnull().sum()
nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
if len(nan_cols):
    for col, cnt in nan_cols.items():
        print(f"  {col:<40} {cnt:>6} NaN")
else:
    print("  None — no NaN remaining")


Before merges        : 3,173 rows
After all merges     : 631 patients, 3,173 rows
NaN counts per column (only columns with NaN):
  ethnicity_source_value                        6 NaN
  race_source_value                             5 NaN


## 10 · Home meds

> `home_meds` is set to `0` — the note parser that extracted this from `Results_UPDATED.csv` is the only part of the original pipeline not reproduced here.

In [10]:
mean_led_per_visit['home_meds'] = 0


## 11 · In-hospital drug one-hot encoding

In [11]:
aggregations = {'generic_name': lambda x: ', '.join(x)}

# Only aggregate PD drugs — mirrors the original TRY_NEW.csv which was PD-only
led_df_pd = led_df[led_df['generic_name'].isin(pd_drug_names)].copy()

result = (
    led_df_pd.set_index('visit_start_datetime')
             .groupby('person_id')
             .resample(TIME_WINDOW)
             .agg(aggregations)
             .reset_index()
)
result.rename(columns={'visit_start_datetime': 'visit_start_datetime'}, inplace=True)

unique_drug_sets = [
    ', '.join(sorted({g for g in row.split(', ')}))
    for row in result['generic_name']
]
patient_drugs = pd.DataFrame({
    'person_id':           result['person_id'].values,
    'visit_start_datetime': result['visit_start_datetime'].values,
    'drugs_per_visit':     unique_drug_sets,
})

drugs_list    = patient_drugs['drugs_per_visit'].str.split(', ')
one_hot_drugs = pd.get_dummies(drugs_list.apply(pd.Series).stack()).groupby(level=0).sum()
one_hot_drugs['person_id']            = patient_drugs['person_id'].values
one_hot_drugs['visit_start_datetime'] = patient_drugs['visit_start_datetime'].values

# Merge by keys — avoids index misalignment from earlier merge() resets
mean_led_per_visit = mean_led_per_visit.merge(
    one_hot_drugs, on=['person_id', 'visit_start_datetime'], how='left'
)
# Forward-filled rows (no drug admin that period) get NaN drug cols → fill with 0
drug_cols = [c for c in one_hot_drugs.columns if c not in ['person_id', 'visit_start_datetime']]
mean_led_per_visit[drug_cols] = mean_led_per_visit[drug_cols].fillna(0)
print(f"Drug one-hot columns added: {len(drug_cols)}  "
      f"(merged by person_id + visit_start_datetime, NaN→0)")


Drug one-hot columns added: 22  (merged by person_id + visit_start_datetime, NaN→0)


## 12 · Final cleanup

In [12]:
mean_led_per_visit.drop(
    columns=['change_in_ledd', 'is_first_visit',
             'percent_change', 'is_change'],
    inplace=True, errors='ignore',
)
# normalized_percent_change is intentionally kept — extracted in section 13
mean_led_per_visit.drop(columns=['person_id', 'visit_start_datetime'], inplace=True)
mean_led_per_visit.dropna(inplace=True)

print(f"Final feature matrix: {mean_led_per_visit.shape}")
print(f"Class balance: {mean_led_per_visit['prediction'].value_counts(normalize=True).to_dict()}")


Final feature matrix: (3166, 36)
Class balance: {0: 0.8177511054958939, 1: 0.18224889450410614}


## 13 · Preprocessing → `xgboost_df`

In [13]:
xgboost_df = mean_led_per_visit.copy()

# Extract regression target NOW — perfectly index-aligned with xgboost_df
normalized_pctg_change = xgboost_df.pop('normalized_percent_change')

demographic_vars = ['gender_source_value', 'race_source_value', 'ethnicity_source_value']
xgboost_df = pd.get_dummies(xgboost_df, columns=[v for v in demographic_vars if v in xgboost_df.columns])

scaler       = MinMaxScaler()
numeric_vars = ['mean_led_per_visit', 'age', 'length_of_stay', 'days_since_last_visit', 'days_to_diagnosis']
for var in [v for v in numeric_vars if v in xgboost_df.columns]:
    xgboost_df[var] = scaler.fit_transform(xgboost_df[[var]])

# Keep prediction as last column
prediction_col = xgboost_df.pop('prediction')
xgboost_df['prediction'] = prediction_col

# Realign regression target after get_dummies (index unchanged, but be explicit)
normalized_pctg_change = normalized_pctg_change.reindex(xgboost_df.index)

X = xgboost_df.iloc[:, :-1]
y = xgboost_df.iloc[:, -1]

print(f"xgboost_df: {xgboost_df.shape}  |  features: {X.shape[1]}")
print(f"normalized_pctg_change NaN count: {normalized_pctg_change.isna().sum()} (should be 0)")


xgboost_df: (3166, 41)  |  features: 40
normalized_pctg_change NaN count: 0 (should be 0)


## 13b · Diagnostic — compare `data.csv` vs reconstruction features

In [14]:
# ═══════════════════════════════════════════════════════════════════
# DIAGNOSTIC: Compare data.csv (original) vs xgboost_df (reconstruction)
# ═══════════════════════════════════════════════════════════════════
original = pd.read_csv('data/data.csv')

# First: check what data.csv actually contains to understand its time window
print("── data.csv quick inspection ──")
print(f"  Rows: {len(original):,}  |  Cols: {len(original.columns)}")
print(f"  mean_led_per_visit → min={original['mean_led_per_visit'].min():.1f}, "
      f"max={original['mean_led_per_visit'].max():.1f}, "
      f"mean={original['mean_led_per_visit'].mean():.1f}")
if 'normalized_percent_change' in original.columns:
    print(f"  normalized_percent_change column: present ✓")
if 'prediction' in original.columns:
    print(f"  Class balance: {original['prediction'].mean():.1%} change / "
          f"{1-original['prediction'].mean():.1%} no-change")
print()

# One-hot encode demographics in original to match xgboost_df preprocessing
demographic_vars = ['gender_source_value', 'race_source_value', 'ethnicity_source_value']
original_ohe = pd.get_dummies(original, columns=[v for v in demographic_vars if v in original.columns])

orig_cols  = set(original_ohe.columns)
recon_cols = set(xgboost_df.columns)

only_in_original     = sorted(orig_cols  - recon_cols)
only_in_recon        = sorted(recon_cols - orig_cols)
in_both              = sorted(orig_cols  & recon_cols)

print(f"{'='*60}")
print(f"  Original (data.csv) columns : {len(orig_cols)}")
print(f"  Reconstruction columns      : {len(recon_cols)}")
print(f"  In both                     : {len(in_both)}")
print(f"  Only in original            : {len(only_in_original)}")
print(f"  Only in reconstruction      : {len(only_in_recon)}")
print(f"{'='*60}")

print(f"\n── Columns ONLY IN ORIGINAL ({len(only_in_original)}) ──")
for c in only_in_original:
    print(f"  {c}")

print(f"\n── Columns ONLY IN RECONSTRUCTION ({len(only_in_recon)}) ──")
for c in only_in_recon:
    print(f"  {c}")

print(f"\n── Shared columns — mean value comparison ──")
print(f"  {'Column':<35} {'Original mean':>14} {'Recon mean':>12} {'Diff':>10}")
print(f"  {'-'*35} {'-'*14} {'-'*12} {'-'*10}")
for c in in_both:
    try:
        om = original_ohe[c].mean()
        rm = xgboost_df[c].mean()
        diff = rm - om
        flag = '  ◄' if abs(diff) > 0.05 else ''
        print(f"  {c:<35} {om:>14.4f} {rm:>12.4f} {diff:>+10.4f}{flag}")
    except Exception:
        print(f"  {c:<35} (non-numeric)")

print(f"\n── Row counts ──")
print(f"  Original rows : {len(original_ohe):,}")
print(f"  Reconstruction: {len(xgboost_df):,}")


── data.csv quick inspection ──
  Rows: 6,105  |  Cols: 35
  mean_led_per_visit → min=0.1, max=3812.9, mean=128.6
  normalized_percent_change column: present ✓
  Class balance: 25.1% change / 74.9% no-change

  Original (data.csv) columns : 41
  Reconstruction columns      : 41
  In both                     : 37
  Only in original            : 4
  Only in reconstruction      : 4

── Columns ONLY IN ORIGINAL (4) ──
  inv levodopa-carbidopa intestinal gel
  istradefylline
  levodopa
  normalized_percent_change

── Columns ONLY IN RECONSTRUCTION (4) ──
  
   rasagiline mesylate
  benztropine mesylate
  has_dbs

── Shared columns — mean value comparison ──
  Column                               Original mean   Recon mean       Diff
  ----------------------------------- -------------- ------------ ----------
  age                                        77.9514       0.6265   -77.3249  ◄
  amantadine                                  0.0981       0.0310    -0.0672  ◄
  amantadine er          

## 14 · Stage 1 — XGBoost Classifier

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
d_train = xgb.DMatrix(X_train, label=y_train)
d_test  = xgb.DMatrix(X_test,  label=y_test)

binary_params = {
    'alpha': 0, 'lambda': 0.1, 'learning_rate': 0.01, 'max_depth': 7,
    'eval_metric': 'auc', 'objective': 'binary:logistic',
    'sampling_method': 'gradient_based', 'tree_method': 'hist', 'device': 'cuda',
}

model_s1 = xgb.train(
    binary_params, d_train,
    num_boost_round=800,
    evals=[(d_test, 'test')],
    verbose_eval=100,
    early_stopping_rounds=50,
)

# Predictions on test set
y_pred_proba = model_s1.predict(d_test, iteration_range=(0, model_s1.best_iteration + 1))
y_pred       = (y_pred_proba > 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)
print(f"\n[Stage 1]  AUC = {auc:.4f}  |  ACC = {acc:.4f}")
print(classification_report(y_test, y_pred))

# Probabilities over the FULL dataset (needed for Stage 2 filtering)
d_all            = xgb.DMatrix(X)
y_pred_proba_all = model_s1.predict(d_all, iteration_range=(0, model_s1.best_iteration + 1))

print(f"\nProb distribution (full dataset):")
for pct in [0, 10, 25, 50, 60, 75, 90, 95, 99, 100]:
    print(f"  p{pct:>3}: {np.percentile(y_pred_proba_all, pct):.4f}")
print(f"  rows > 0.5 : {(y_pred_proba_all > 0.50).sum():,}")
print(f"  rows > 0.6 : {(y_pred_proba_all > 0.60).sum():,}")
print(f"  rows > 0.75: {(y_pred_proba_all > 0.75).sum():,}")
print(f"\nClass balance in full dataset:")
print(f"  y=1 (change)   : {int(y.sum()):,} ({y.mean():.1%})")
print(f"  y=0 (no change): {int((1-y).sum()):,} ({(1-y).mean():.1%})")

# ── Youden index threshold (derived from this model's ROC on the test split) ──
from sklearn.metrics import roc_curve
fpr_s1, tpr_s1, thresholds_s1 = roc_curve(y_test, y_pred_proba)
youden_idx     = np.argmax(tpr_s1 - fpr_s1)
STAGE2_CUTOFF  = float(thresholds_s1[youden_idx])
print(f"\nYouden index threshold (this model): {STAGE2_CUTOFF:.4f}")
print(f"  rows in Stage 2 pool at this threshold: {(y_pred_proba_all > STAGE2_CUTOFF).sum():,} "      f"({(y_pred_proba_all > STAGE2_CUTOFF).mean():.1%} of dataset)")


[0]	test-auc:0.92282
[99]	test-auc:0.92839

[Stage 1]  AUC = 0.9286  |  ACC = 0.8328
              precision    recall  f1-score   support

           0       0.83      1.00      0.91       521
           1       0.89      0.07      0.13       113

    accuracy                           0.83       634
   macro avg       0.86      0.53      0.52       634
weighted avg       0.84      0.83      0.77       634


Prob distribution (full dataset):
  p  0: 0.1116
  p 10: 0.1116
  p 25: 0.1116
  p 50: 0.1116
  p 60: 0.1186
  p 75: 0.2307
  p 90: 0.4166
  p 95: 0.4398
  p 99: 0.5038
  p100: 0.5107
  rows > 0.5 : 46
  rows > 0.6 : 0
  rows > 0.75: 0

Class balance in full dataset:
  y=1 (change)   : 577 (18.2%)
  y=0 (no change): 2,589 (81.8%)

Youden index threshold (this model): 0.1645
  rows in Stage 2 pool at this threshold: 1,016 (32.1% of dataset)


## 15 · Stage 2 — XGBoost Regressor

In [16]:
# Filter to samples where Stage 1 predicts a LEDD change with p > cutoff
# Use a pandas Series mask (aligned by index) to avoid positional mismatch
# on the gapped index that results from dropna().
xgboost_df['prediction'] = normalized_pctg_change.values

# STAGE2_CUTOFF was derived from the Youden index in the Stage 1 cell above.
# Using a Series (not numpy array) ensures correct alignment on the gapped
# post-dropna index.
prob_series = pd.Series(y_pred_proba_all, index=xgboost_df.index)
filtered_df = xgboost_df[prob_series > STAGE2_CUTOFF]

print(f"Cutoff (Youden, this model): {STAGE2_CUTOFF:.4f}")
print(f"Stage 2 pool: {len(filtered_df):,} rows  ({len(filtered_df)/len(xgboost_df):.1%} of dataset)")

X2 = filtered_df.iloc[:, :-1]
y2 = filtered_df.iloc[:, -1]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=RANDOM_STATE
)
d2_train = xgb.DMatrix(X2_train, label=y2_train)
d2_test  = xgb.DMatrix(X2_test,  label=y2_test)

regression_params = {
    'alpha': 0.1, 'lambda': 1, 'learning_rate': 0.1, 'max_depth': 7,
    'eval_metric': 'rmse', 'objective': 'reg:squarederror',
    'sampling_method': 'gradient_based', 'tree_method': 'hist', 'device': 'cuda',
}

model_s2 = xgb.train(
    regression_params, d2_train,
    num_boost_round=700,
    evals=[(d2_train, 'train'), (d2_test, 'test')],
    verbose_eval=100,
    early_stopping_rounds=10,
)

y2_pred = model_s2.predict(d2_test, iteration_range=(0, model_s2.best_iteration + 1))
rmse = np.sqrt(mean_squared_error(y2_test, y2_pred))
mae  = mean_absolute_error(y2_test, y2_pred)
r2   = r2_score(y2_test, y2_pred)
print(f"\n[Stage 2]  RMSE = {rmse:.4f}  |  MAE = {mae:.4f}  |  R² = {r2:.4f}")


Cutoff (Youden, this model): 0.1645
Stage 2 pool: 1,016 rows  (32.1% of dataset)
[0]	train-rmse:0.42687	test-rmse:0.45072
[19]	train-rmse:0.24633	test-rmse:0.43894

[Stage 2]  RMSE = 0.4298  |  MAE = 0.2951  |  R² = 0.1125


## 16 · Summary

In [17]:
print("=" * 45)
print(f"  Stage 1  AUC  : {auc:.4f}")
print(f"  Stage 1  ACC  : {acc:.4f}")
print(f"  Stage 2  RMSE : {rmse:.4f}")
print(f"  Stage 2  MAE  : {mae:.4f}")
print(f"  Stage 2  R²   : {r2:.4f}")
print("=" * 45)
print("Compare these against your original results.")
print("Only expected difference: home_meds=0 for all patients.")


  Stage 1  AUC  : 0.9286
  Stage 1  ACC  : 0.8328
  Stage 2  RMSE : 0.4298
  Stage 2  MAE  : 0.2951
  Stage 2  R²   : 0.1125
Compare these against your original results.
Only expected difference: home_meds=0 for all patients.


In [18]:
# How many unique drug_exposure_start_datetime values does TRY_NEW.csv have per patient?
print(f"led_df shape: {led_df.shape}")
print(f"Columns: {list(led_df.columns)}")
print()
unique_drug_dates = led_df.groupby('person_id')['drug_exposure_start_datetime'].nunique()
print(f"Unique drug_exposure_start_datetime per patient:")
print(f"  min={unique_drug_dates.min()}  median={unique_drug_dates.median():.0f}  max={unique_drug_dates.max()}")
print()
# Sample one patient to see their dates
sample_pt = led_df['person_id'].iloc[0]
sample_dates = led_df[led_df['person_id'] == sample_pt]['drug_exposure_start_datetime'].sort_values().unique()
print(f"Sample patient {sample_pt} has {len(sample_dates)} unique dates:")
print(sample_dates[:10])

led_df shape: (554590, 16)
Columns: ['person_id', 'drug_source_value', 'drug_exposure_start_datetime', 'drug_info', 'generic_name', 'brand_name', 'dosage', 'led_dose', 'visit_occurrence_id', 'visit_concept_id', 'visit_start_datetime', 'dose_source_value', 'dose_unit_source_value', 'route_source_value', 'visit_detail_id', 'led']

Unique drug_exposure_start_datetime per patient:
  min=9  median=229  max=10749

Sample patient 158166 has 198 unique dates:
<DatetimeArray>
['2011-05-16 07:45:00', '2011-05-16 11:07:00', '2011-05-16 14:48:00',
 '2011-05-16 15:30:00', '2011-05-16 17:30:00', '2011-05-16 21:30:00',
 '2011-05-16 22:45:00', '2011-05-16 23:45:00', '2011-05-17 05:43:00',
 '2011-05-17 06:30:00']
Length: 10, dtype: datetime64[ns]


In [19]:
# Load original target from data.csv
original_df = pd.read_csv('data/data.csv')
orig_npc = original_df['normalized_percent_change'].dropna()

# Reconstruction target
recon_npc = normalized_pctg_change.dropna()

print("── normalized_percent_change comparison ──")
print(f"  Original  → count={len(orig_npc):,}  mean={orig_npc.mean():.4f}  "
      f"std={orig_npc.std():.4f}  min={orig_npc.min():.4f}  max={orig_npc.max():.4f}")
print(f"  Recon     → count={len(recon_npc):,}  mean={recon_npc.mean():.4f}  "
      f"std={recon_npc.std():.4f}  min={recon_npc.min():.4f}  max={recon_npc.max():.4f}")
print()

# Value distribution
print("  Original percentiles:")
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"    p{p:>3}: {np.percentile(orig_npc, p):.4f}")
print()
print("  Reconstruction percentiles:")
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"    p{p:>3}: {np.percentile(recon_npc, p):.4f}")
print()

# Zero proportion (no-change rows)
print(f"  Original  → zeros: {(orig_npc == 0).mean():.1%}")
print(f"  Recon     → zeros: {(recon_npc == 0).mean():.1%}")

── normalized_percent_change comparison ──
  Original  → count=6,105  mean=0.0052  std=0.3507  min=-1.0000  max=1.0000
  Recon     → count=3,166  mean=0.0032  std=0.2585  min=-1.0000  max=1.0000

  Original percentiles:
    p  0: -1.0000
    p 10: -0.3200
    p 25: 0.0000
    p 50: 0.0000
    p 75: 0.0000
    p 90: 0.4900
    p100: 1.0000

  Reconstruction percentiles:
    p  0: -1.0000
    p 10: 0.0000
    p 25: 0.0000
    p 50: 0.0000
    p 75: 0.0000
    p 90: 0.1400
    p100: 1.0000

  Original  → zeros: 74.9%
  Recon     → zeros: 81.8%
